In [175]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering
from sklearn.ensemble import RandomForestRegressor


In [176]:
dftrain = pd.read_csv('train.csv')
dftest = pd.read_csv('test.csv')

In [177]:
dftrain.isnull().sum()

SampleID                0
canvas_size             0
is_oil_painting         0
brush_type              0
num_colors              0
colorfulness            0
complexity              0
brightness              0
contrast                0
stroke_density          0
has_signature           0
is_framed               0
uses_gold_leaf          0
is_restored             0
dominant_warm_colors    0
dominant_color          0
art_period_hint         0
auction_house           0
image_quality           0
brightness_log          0
complexity_x_stroke     0
fake_style_score        0
painter_style_score     0
target_price            0
dtype: int64

In [178]:
def scor_sub1(row):
    score = 0 

    if row['stroke_density'] > 0.7:
        score += 2

    if row['complexity'] > 0.65:
        score += 2

    if row['uses_gold_leaf']:
        score += 1

    if row['has_signature']:
        score += 1

    if (row['num_colors'] > 65) and (row['colorfulness'] > 0.7):
        score += 2

    if (row['contrast'] < 0.4) or (row['brightness'] < 0.45) or (row['brightness'] > 0.75):
        score -= 1

    return score

In [179]:
dftest['SAA'] = dftest.apply(scor_sub1, axis=1 )
sub1 = dftest['SAA'].apply(lambda x: 'Autentic' if x >=5 else 'Incert')

In [180]:
sub1

0        Incert
1      Autentic
2        Incert
3        Incert
4        Incert
         ...   
235      Incert
236      Incert
237      Incert
238      Incert
239    Autentic
Name: SAA, Length: 240, dtype: object

In [181]:
dftrain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   SampleID              960 non-null    int64  
 1   canvas_size           960 non-null    object 
 2   is_oil_painting       960 non-null    bool   
 3   brush_type            960 non-null    object 
 4   num_colors            960 non-null    int64  
 5   colorfulness          960 non-null    float64
 6   complexity            960 non-null    float64
 7   brightness            960 non-null    float64
 8   contrast              960 non-null    float64
 9   stroke_density        960 non-null    float64
 10  has_signature         960 non-null    bool   
 11  is_framed             960 non-null    bool   
 12  uses_gold_leaf        960 non-null    bool   
 13  is_restored           960 non-null    bool   
 14  dominant_warm_colors  960 non-null    bool   
 15  dominant_color        9

In [182]:
dftrain


,SampleID,canvas_size,is_oil_painting,brush_type,num_colors,colorfulness,complexity,brightness,contrast,stroke_density,...,dominant_warm_colors,dominant_color,art_period_hint,auction_house,image_quality,brightness_log,complexity_x_stroke,fake_style_score,painter_style_score,target_price
0,332,60x50,True,medium,71,0.616240,0.755582,0.647338,0.587923,0.702118,...,False,red,baroque,Online,low,0.499161,0.530508,0.349718,0.536437,50800
1,410,80x90,True,medium,60,0.660715,0.474923,0.538822,0.599076,0.528112,...,False,yellow,modern,Online,low,0.431017,0.250813,0.258722,0.163906,37400
2,77,80x50,True,fine,64,0.684877,0.380591,0.608029,0.500152,0.508521,...,False,mixed,baroque,Sothebys,low,0.475009,0.193539,0.797662,0.137732,35500
3,869,80x50,True,medium,56,0.427938,0.581636,0.562086,0.483896,0.550152,...,False,blue,modern,Local,low,0.446022,0.319988,0.569981,0.394542,43200
4,139,80x130,True,fine,55,0.481406,0.629780,0.476093,0.493429,0.681710,...,False,red,modern,Local,medium,0.389398,0.429328,0.723147,0.321858,54500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,1045,60x90,True,medium,67,0.761182,0.740730,0.747183,0.635812,0.871171,...,True,yellow,modern,Local,low,0.558005,0.645302,0.356524,0.734686,63200
956,1096,100x70,True,mixed,44,0.406706,0.574378,0.558835,0.566975,0.746025,...,False,blue,surrealism,Sothebys,medium,0.443938,0.428500,0.619575,0.362523,49400
957,1131,80x110,True,medium,62,0.594876,0.715957,0.601200,0.548615,0.725513,...,False,blue,baroque,Sothebys,medium,0.470754,0.519436,0.442087,0.956256,72900
958,861,80x50,True,mixed,64,0.599719,0.732753,0.500665,0.532287,0.674759,...,False,mixed,baroque,Local,high,0.405908,0.494432,0.328414,0.952113,62600


In [183]:
    dftrain[['width', 'height']] = dftrain['canvas_size'].str.split('x', expand=True)
    dftrain = dftrain.drop('canvas_size')
    dftrain['width'] = dftrain['width'].astype(float)
    dftrain['height'] = dftrain['height'].astype(float)

    dftest[['width', 'height']] = dftest['canvas_size'].str.split('x', expand=True)
    dftest = dftest.drop('canvas_size')
    dftest['width'] = dftest['width'].astype(float)
    dftest['height'] = dftest['height'].astype(float)


KeyError: "['canvas_size'] not found in axis"

In [ ]:
def clean(df):
    df[['width', 'height']] = df['canvas_size'].str.split('x', expand=True)
    df = df.drop('canvas_size')
    df['width'] = df['width'].astype(float)
    df['height'] = df['height'].astype(float)

    cols_bool = df.select_dtypes(include='bool').columns
    df[cols_bool] = df[cols_bool].astype(int)

    df = pd.get_dummies(df, drop_first=True)
    cols_bool = df.select_dtypes(include='bool').columns
    df[cols_bool] = df[cols_bool].astype(int)    
    return df

In [ ]:
dftrain = clean(dftrain)
dftest = clean(dftest)

KeyError: "['canvas_size'] not found in axis"

In [ ]:
dftrain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 60 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   SampleID                       960 non-null    int64  
 1   is_oil_painting                960 non-null    int64  
 2   num_colors                     960 non-null    int64  
 3   colorfulness                   960 non-null    float64
 4   complexity                     960 non-null    float64
 5   brightness                     960 non-null    float64
 6   contrast                       960 non-null    float64
 7   stroke_density                 960 non-null    float64
 8   has_signature                  960 non-null    int64  
 9   is_framed                      960 non-null    int64  
 10  uses_gold_leaf                 960 non-null    int64  
 11  is_restored                    960 non-null    int64  
 12  dominant_warm_colors           960 non-null    int